# Week 3 — Quality gate and gateway profile

This notebook runs the new Week 3 tools in the correct order: quality first, then a small gateway benchmark, then the controlled pressure case.

**Before starting:** run the Week 1 model server on port 8001 and the Week 2 gateway on port 8000 in two terminals. The notebook calls only the gateway.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import urllib.request

ROOT = Path.cwd().resolve()
if not (ROOT / 'week-03-profile-and-evaluate').exists():
    ROOT = ROOT.parent
WEEK3 = ROOT / 'week-03-profile-and-evaluate'
RESULTS = WEEK3 / 'results'

def run_week3(script, *args):
    command = [sys.executable, str(WEEK3 / script), *map(str, args)]
    print(' '.join(command))
    return subprocess.run(command, cwd=ROOT, check=True)

## 1. Confirm the public gateway is reachable

The benchmark must use the public gateway, not the private Week 1 model-server URL.

In [ ]:
with urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=5) as response:
    print(json.loads(response.read()))

## 2. Freeze the quality baseline

These are ten different fixed cases. Run them twice. The second run compares its pass/fail outcome for every case with the first. A failed case is evidence to inspect; it does not mean the script itself has failed.

In [ ]:
run_week3('quality_gate.py', '--label', 'baseline-run-1')
run_week3(
    'quality_gate.py',
    '--label', 'baseline-run-2',
    '--compare-with', RESULTS / 'quality-baseline-run-1.json',
)

In [ ]:
quality = json.loads((RESULTS / 'quality-baseline-run-2.json').read_text())
print('quality score:', quality['score']['passed'], '/', quality['score']['total'])
[(item['id'], item['passed'], item['failures']) for item in quality['score']['results'] if not item['passed']]

## 3. Smoke-test the benchmark client

Run one short, low-risk streaming wave first. It writes raw request rows and a metrics artifact. TTFT and total latency are client-visible measurements; queue time remains unavailable until the gateway has a real queue timer.

In [ ]:
run_week3(
    'benchmark_client.py',
    '--mode', 'matrix',
    '--prompt-lengths', '128',
    '--concurrencies', '1',
    '--warmup-waves', '1',
    '--measured-waves', '1',
    '--max-tokens', '16',
    '--label', 'smoke',
)

## 4. Run the full matrix

A **wave** means one simultaneous group. At concurrency 16, one measured wave records 16 request rows. Start smaller if you are learning the tool, then use the full 2 warm-up / 10 measured-wave plan for the final baseline.

In [ ]:
RUN_FULL_MATRIX = False

if RUN_FULL_MATRIX:
    run_week3('benchmark_client.py', '--mode', 'matrix', '--label', 'baseline')
else:
    print('Set RUN_FULL_MATRIX = True after the smoke test is understood.')

## 5. Controlled pressure case

This is one 4K-context wave with 16 simultaneous requests. It is not a crash test. Stop if the computer becomes unresponsive or errors repeat. The output tells you the observed boundary: successes, failures, TTFT, total latency, peak in-flight requests, and backend errors.

In [ ]:
RUN_PRESSURE_CASE = False

if RUN_PRESSURE_CASE:
    run_week3('benchmark_client.py', '--mode', 'pressure', '--label', 'pressure-4k-c16')
else:
    print('Set RUN_PRESSURE_CASE = True only after the full matrix has completed.')

## 6. Write the conclusion

Copy the reported p50/p95 values into `BOTTLENECK-REPORT.md`. Separate facts from inference: for example, *4K prompts increased client-observed TTFT* is a fact; *prefill is the likely cause* is an inference. Do not claim a queue-time number when the current gateway cannot measure one.